# Flight & Weather Parquet File Aggregation

This notebook can be used to analyze parquet files for this project

In [1]:
import pandas as pd
import os
import re
from pathlib import Path

In [4]:
# 1. Define the path to your Parquet file
#file_path = r"C:\Users\nefis\OneDrive\Desktop\aegean-flight-reliability-airline-tiers\data\raw\weather_routes\EDDF_LGTS\weather_EDDF_LGTS.parquet"
file_path = Path("example_data/weather_EDDF_LGTS.parquet")

try:
    # 2. Read the Parquet file into a pandas DataFrame
    df = pd.read_parquet(file_path, engine="pyarrow")

    # 3. Print the first 5 rows (Data Preview)
    print("--- FIRST 5 ROWS ---")
    print(df.head())
    print("\n" + "="*50 + "\n")

    # 4. Print column names, data types, and missing values count
    print("--- DATAFRAME INFORMATION ---")
    print(df.info())
    print("\n" + "="*50 + "\n")

    # 5. Print summary statistics for numerical columns
    print("--- SUMMARY STATISTICS ---")
    print(df.describe())
    print("\n" + "="*50 + "\n")
    
    # 6. Print total row and column counts
    print(f"Total Rows: {df.shape[0]} | Total Columns: {df.shape[1]}")

except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found. Please check the path.")
except Exception as e:
    print(f"An error occurred: {e}")

--- FIRST 5 ROWS ---
                       time  temperature_2m  relative_humidity_2m  \
0 2023-07-31 00:00:00+00:00            17.0                    76   
1 2023-07-31 00:00:00+00:00            23.6                    74   
2 2023-07-31 01:00:00+00:00            16.9                    77   
3 2023-07-31 01:00:00+00:00            23.0                    75   
4 2023-07-31 02:00:00+00:00            16.5                    80   

   dew_point_2m  precipitation  weather_code  pressure_msl  cloud_cover_low  \
0          12.8            0.1            61        1015.3               22   
1          18.7            0.0             2        1007.9                0   
2          12.8            0.2            61        1014.8               18   
3          18.4            0.0             0        1007.7                0   
4          13.0            0.5            61        1014.6               60   

   cloud_cover_high  visibility  wind_speed_10m  wind_speed_180m  \
0               100  

In [3]:
# 1. Define the path to your Parquet file
file_path = r"C:\Users\nefis\OneDrive\Desktop\aegean-flight-reliability-airline-tiers\data\raw\EDDF_LGTS\arrivals_LGTS_2024-07-30_2024-07-30.parquet"

try:
    # 2. Read the Parquet file into a pandas DataFrame
    df = pd.read_parquet(file_path, engine="pyarrow")

    # 3. Print the first 5 rows (Data Preview)
    print("--- FIRST 5 ROWS ---")
    print(df.head())
    print("\n" + "="*50 + "\n")

    # 4. Print column names, data types, and missing values count
    print("--- DATAFRAME INFORMATION ---")
    print(df.info())
    print("\n" + "="*50 + "\n")

    # 5. Print summary statistics for numerical columns
    print("--- SUMMARY STATISTICS ---")
    print(df.describe())
    print("\n" + "="*50 + "\n")
    
    # 6. Print total row and column counts
    print(f"Total Rows: {df.shape[0]} | Total Columns: {df.shape[1]}")

except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found. Please check the path.")
except Exception as e:
    print(f"An error occurred: {e}")

--- FIRST 5 ROWS ---
   icao24   firstSeen estDepartureAirport    lastSeen estArrivalAirport  \
0  46bc4a  1722374282                LGKO  1722379024              LGTS   
1  46bc4d  1722374974                LGMK  1722378429              LGTS   
2  4ca740  1722372434                 NaN  1722378150              LGTS   
3  4ca84f  1722364110                EFHK  1722374198              LGTS   
4  4691c3  1722355831                LGTS  1722372347              LGTS   

   callsign  estDepartureAirportHorizDistance  \
0  AEE597                              1033.0   
1  AEE593                              8765.0   
2  RYR5MJ                                 NaN   
3  RYR9229                             1910.0   
4  AEE544                               273.0   

   estDepartureAirportVertDistance  estArrivalAirportHorizDistance  \
0                            209.0                            2442   
1                            806.0                            1919   
2                      

In [4]:

# 1. Define the input directory and output file paths
INPUT_DIR = Path(r"C:\Users\nefis\OneDrive\Desktop\aegean-flight-reliability-airline-tiers\data\raw\EDDF_LGTS")
# Use the exact path causing the issue
OUTPUT_FILE = Path(r"C:\Users\nefis\OneDrive\Desktop\aegean-flight-reliability-airline-tiers\data\processed\test\combined_arrivals_LGTS.parquet")


def combine_parquet_files():
    df_list = []

    if not INPUT_DIR.exists():
        print(f"Error: The directory '{INPUT_DIR}' does not exist.")
        return

    # 2. Loop through all parquet files matching the pattern
    print("Processing files...")
    for file_path in INPUT_DIR.glob("arrivals_LGTS_*.parquet"):
        match = re.search(r"arrivals_LGTS_(\d{4}-\d{2}-\d{2})", file_path.name)

        if match:
            extracted_date = match.group(1)

            try:
                df = pd.read_parquet(file_path, engine="pyarrow")
                df["flight_date"] = extracted_date
                df_list.append(df)
                print(f"Successfully processed: {file_path.name} ({extracted_date})")

            except Exception as e:
                print(f"Error reading {file_path.name}: {e}")

    # 3. Combine all dataframes and save to a new file
    if df_list:
        print("\nCombining all dataframes...")
        combined_df = pd.concat(df_list, ignore_index=True)

        # --- FIX: Automatically create the folder path if it does not exist ---
        output_dir = OUTPUT_FILE.parent
        if not output_dir.exists():
            print(f"Creating missing directory path: {output_dir}")
            output_dir.mkdir(parents=True, exist_ok=True)
        # ---------------------------------------------------------------------

        print(f"Saving combined data to {OUTPUT_FILE}...")
        combined_df.to_parquet(OUTPUT_FILE, engine="pyarrow", index=False)

        print(f"\nSuccess! Combined {len(df_list)} files.")
        print(f"Total rows in combined file: {len(combined_df)}")
    else:
        print("\nNo valid Parquet files were found or processed.")


if __name__ == "__main__":
    combine_parquet_files()


Processing files...
Successfully processed: arrivals_LGTS_2023-07-31_2023-07-31.parquet (2023-07-31)
Successfully processed: arrivals_LGTS_2023-08-01_2023-08-01.parquet (2023-08-01)
Successfully processed: arrivals_LGTS_2023-08-02_2023-08-02.parquet (2023-08-02)
Successfully processed: arrivals_LGTS_2023-08-03_2023-08-03.parquet (2023-08-03)
Successfully processed: arrivals_LGTS_2023-08-04_2023-08-04.parquet (2023-08-04)
Successfully processed: arrivals_LGTS_2023-08-05_2023-08-05.parquet (2023-08-05)
Successfully processed: arrivals_LGTS_2023-08-06_2023-08-06.parquet (2023-08-06)
Successfully processed: arrivals_LGTS_2023-08-07_2023-08-07.parquet (2023-08-07)
Successfully processed: arrivals_LGTS_2023-08-08_2023-08-08.parquet (2023-08-08)
Successfully processed: arrivals_LGTS_2023-08-09_2023-08-09.parquet (2023-08-09)
Successfully processed: arrivals_LGTS_2023-08-10_2023-08-10.parquet (2023-08-10)
Successfully processed: arrivals_LGTS_2023-08-11_2023-08-11.parquet (2023-08-11)
Successf

In [5]:
# 1. Define the path to your Parquet file
file_path = Path(r"C:\Users\nefis\OneDrive\Desktop\aegean-flight-reliability-airline-tiers\data\processed\test\combined_arrivals_LGTS.parquet")

try:
    # 2. Read the Parquet file into a pandas DataFrame
    df = pd.read_parquet(file_path, engine="pyarrow")

    # 3. Print the first 5 rows (Data Preview)
    print("--- FIRST 5 ROWS ---")
    print(df.head())
    print("\n" + "="*50 + "\n")

    # 4. Print column names, data types, and missing values count
    print("--- DATAFRAME INFORMATION ---")
    print(df.info())
    print("\n" + "="*50 + "\n")

    # 5. Print summary statistics for numerical columns
    print("--- SUMMARY STATISTICS ---")
    print(df.describe())
    print("\n" + "="*50 + "\n")
    
    # 6. Print total row and column counts
    print(f"Total Rows: {df.shape[0]} | Total Columns: {df.shape[1]}")

except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found. Please check the path.")
except Exception as e:
    print(f"An error occurred: {e}")

--- FIRST 5 ROWS ---
   icao24     firstSeen estDepartureAirport      lastSeen estArrivalAirport  \
0  4ca8d6  1.690836e+09                EDDB  1.690843e+09              LGTS   
1  46b927  1.690841e+09                LGAV  1.690842e+09              LGTS   
2  46b8a3  1.690840e+09                LGAV  1.690842e+09              LGTS   
3  46bc4f  1.690836e+09                LGRP  1.690841e+09              LGTS   
4  46bc4c  1.690835e+09                 NaN  1.690839e+09              LGTS   

   callsign  estDepartureAirportHorizDistance  \
0  RYR41AU                             2578.0   
1  SEH6HG                              6779.0   
2  AEE136                              6895.0   
3  AEE4993                              883.0   
4  OAL085                                 NaN   

   estDepartureAirportVertDistance  estArrivalAirportHorizDistance  \
0                             74.0                          2605.0   
1                           1178.0                          2684.0   

In [6]:

# 1. Define your file paths
INPUT_FILE = Path(r"C:\Users\nefis\OneDrive\Desktop\aegean-flight-reliability-airline-tiers\data\processed\test\combined_arrivals_LGTS.parquet")
OUTPUT_FILE = Path(r"C:\Users\nefis\OneDrive\Desktop\aegean-flight-reliability-airline-tiers\data\processed\test\combined_arrivals_LGTS_utc.parquet")


# Define which columns contain the Unix timestamps
FIRST_SEEN_COL = "firstSeen"
LAST_SEEN_COL = "lastSeen"
DEST_AIRPORT_COL = "estDepartureAirport"  # Column tracking departure codes


def convert_and_find_flight_timestamps():
    try:
        # 2. Read the parquet file
        print(f"Reading {INPUT_FILE}...")
        df = pd.read_parquet(INPUT_FILE, engine="pyarrow")

        # Check if required columns exist
        required_cols = [FIRST_SEEN_COL, LAST_SEEN_COL, DEST_AIRPORT_COL]
        missing_cols = [col for col in required_cols if col not in df.columns]
        if missing_cols:
            print(f"Error: Columns {missing_cols} not found in the file.")
            print(f"Available columns: {list(df.columns)}")
            return

        print("Converting timestamps and generating hourly columns...")

        # FIX: Dynamically detect and handle the incoming format safely
        for col in [FIRST_SEEN_COL, LAST_SEEN_COL]:
            # Fill missing data safely before any translation steps
            df[col] = df[col].fillna(0)
            
            # If the parquet already loaded it as a datetime, drop it back to raw integer format
            if pd.api.types.is_datetime64_any_dtype(df[col]):
                df[col] = df[col].astype("int64")
            else:
                df[col] = df[col].astype("int64")

            # Check scale: If the number is huge (19 digits long), it's nanoseconds. 
            # If it's a normal 10-digit number (starts with 16 or 17), it's seconds.
            sample_val = df[col].iloc[0] if len(df) > 0 else 0
            if sample_val > 10**11:  # It's in nanoseconds or microseconds scale
                # Safe conversion converting from nanoseconds unit natively
                df[col] = pd.to_datetime(df[col], unit="ns", utc=True)
            else:
                # Normal conversion from standard seconds unit
                df[col] = pd.to_datetime(df[col], unit="s", utc=True)

        # 5. Generate new rounded hourly columns cleanly using native pandas rounder
        df["firstSeen_hourly_utc"] = df[FIRST_SEEN_COL].dt.round("h")
        df["lastSeen_hourly_utc"] = df[LAST_SEEN_COL].dt.round("h")

        # 6. Filter for EDDF destination routes only
        print("Filtering dataset for destination 'EDDF'...")
        df = df[df[DEST_AIRPORT_COL] == "EDDF"]

        if df.empty:
            print("Warning: No flights found with estDestinationAirport 'EDDF'. Output file will be empty.")

        # 7. Ensure output directory exists and save the file
        OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
        print(f"Saving filtered file to {OUTPUT_FILE} ({len(df)} rows)...")
        df.to_parquet(OUTPUT_FILE, engine="pyarrow", index=False)

        print("\nSuccess!")
        preview_cols = [
            DEST_AIRPORT_COL,
            FIRST_SEEN_COL,
            LAST_SEEN_COL,
            "firstSeen_hourly_utc",
            "lastSeen_hourly_utc",
        ]
        print("\nData Preview:")
        print(df[preview_cols].head())

    except FileNotFoundError:
        print(f"Error: File '{INPUT_FILE}' not found.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")


if __name__ == "__main__":
    convert_and_find_flight_timestamps()



Reading C:\Users\nefis\OneDrive\Desktop\aegean-flight-reliability-airline-tiers\data\processed\test\combined_arrivals_LGTS.parquet...
Converting timestamps and generating hourly columns...
Filtering dataset for destination 'EDDF'...
Saving filtered file to C:\Users\nefis\OneDrive\Desktop\aegean-flight-reliability-airline-tiers\data\processed\test\combined_arrivals_LGTS_utc.parquet (2081 rows)...

Success!

Data Preview:
    estDepartureAirport                 firstSeen                  lastSeen  \
34                 EDDF 2023-07-31 13:00:41+00:00 2023-07-31 14:55:19+00:00   
53                 EDDF 2023-07-31 10:25:08+00:00 2023-07-31 12:16:24+00:00   
142                EDDF 2023-08-01 13:05:03+00:00 2023-08-01 14:53:54+00:00   
155                EDDF 2023-08-01 10:21:02+00:00 2023-08-01 12:15:58+00:00   
228                EDDF 2023-08-02 13:16:52+00:00 2023-08-02 15:05:57+00:00   

         firstSeen_hourly_utc       lastSeen_hourly_utc  
34  2023-07-31 13:00:00+00:00 2023-07-31 15

In [7]:
# 1. Define the path to your Parquet file
file_path = r"C:\Users\nefis\OneDrive\Desktop\aegean-flight-reliability-airline-tiers\data\raw\weather_routes\EDDF_LGTS\weather_EDDF_LGTS.parquet"

try:
    # 2. Read the Parquet file into a pandas DataFrame
    df = pd.read_parquet(file_path, engine="pyarrow")

    # 3. Print the first 5 rows (Data Preview)
    print("--- FIRST 5 ROWS ---")
    print(df.head())
    print("\n" + "="*50 + "\n")

    # 4. Print column names, data types, and missing values count
    print("--- DATAFRAME INFORMATION ---")
    print(df.info())
    print("\n" + "="*50 + "\n")

    # 5. Print summary statistics for numerical columns
    print("--- SUMMARY STATISTICS ---")
    print(df.describe())
    print("\n" + "="*50 + "\n")
    
    # 6. Print total row and column counts
    print(f"Total Rows: {df.shape[0]} | Total Columns: {df.shape[1]}")

except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found. Please check the path.")
except Exception as e:
    print(f"An error occurred: {e}")

--- FIRST 5 ROWS ---
                       time  temperature_2m  relative_humidity_2m  \
0 2023-07-31 00:00:00+00:00            17.0                    76   
1 2023-07-31 00:00:00+00:00            23.6                    74   
2 2023-07-31 01:00:00+00:00            16.9                    77   
3 2023-07-31 01:00:00+00:00            23.0                    75   
4 2023-07-31 02:00:00+00:00            16.5                    80   

   dew_point_2m  precipitation  weather_code  pressure_msl  cloud_cover_low  \
0          12.8            0.1            61        1015.3               22   
1          18.7            0.0             2        1007.9                0   
2          12.8            0.2            61        1014.8               18   
3          18.4            0.0             0        1007.7                0   
4          13.0            0.5            61        1014.6               60   

   cloud_cover_high  visibility  wind_speed_10m  wind_speed_180m  \
0               100  

In [8]:

weather_path = r"C:\Users\nefis\OneDrive\Desktop\aegean-flight-reliability-airline-tiers\data\raw\weather_routes\EDDF_LGTS\weather_EDDF_LGTS.parquet"
flights_path = r"C:\Users\nefis\OneDrive\Desktop\aegean-flight-reliability-airline-tiers\data\processed\test\combined_arrivals_LGTS_utc.parquet"

w_df = pd.read_parquet(weather_path, engine="pyarrow")
f_df = pd.read_parquet(flights_path, engine="pyarrow")

print(w_df.info())
print(f_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 52656 entries, 0 to 52655
Data columns (total 18 columns):
 #   Column                      Non-Null Count  Dtype              
---  ------                      --------------  -----              
 0   time                        52656 non-null  datetime64[us, UTC]
 1   temperature_2m              52656 non-null  float64            
 2   relative_humidity_2m        52656 non-null  int64              
 3   dew_point_2m                52656 non-null  float64            
 4   precipitation               52656 non-null  float64            
 5   weather_code                52656 non-null  int64              
 6   pressure_msl                52656 non-null  float64            
 7   cloud_cover_low             52656 non-null  int64              
 8   cloud_cover_high            52656 non-null  int64              
 9   visibility                  52656 non-null  float64            
 10  wind_speed_10m              52656 non-null  float64            
 11  

In [9]:
print(w_df.duplicated(["airport", "time"]).sum())

0
